<a href="https://colab.research.google.com/github/theelderemo/wan2.2-google-colab/blob/main/wan2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wan2.2 — Complete Video Generation Suite

This notebook covers **every generation mode** in Wan2.2:

| # | Mode | Model | Notes |
|---|------|-------|-------|
| 1 | **Text-to-Video** | T2V-A14B (MoE 27B / 14B active) | 480P & 720P |
| 2 | **Image-to-Video** | I2V-A14B (MoE 27B / 14B active) | 480P & 720P |
| 3 | **Text+Image-to-Video** | TI2V-5B | 720P @ 24 FPS, runs on 4090 |
| 4 | **Speech-to-Video** | S2V-14B | Audio + image → talking-head video |
| 4a | **Speech-to-Video + Pose** | S2V-14B | Pose-driven with audio |
| 4b | **Speech-to-Video + TTS** | S2V-14B + CosyVoice | Generate speech then animate |
| 5 | **Character Animation** | Animate-14B | Mimic human motion from video |
| 6 | **Character Replacement** | Animate-14B | Replace character in video |

**Workflow:** Run **Section 0** (setup) once → **Section 1** (download only the models you need) → jump to any generation section.

---
**GPU requirements at a glance:**
- TI2V-5B (Section 3): ≥ 24 GB VRAM (RTX 4090 works)
- All other models: ≥ 80 GB VRAM (A100/H100), or use `offload_model + convert_model_dtype` flags
- Colab A100 recommended for the A14B models

---
## Section 0 — Setup & Installation

Run these cells **once per Colab session**.

In [2]:
# ── 0.1  Check GPU ────────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU detected — check Colab runtime type.")

Sat Jun 27 23:43:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ── 0.2  Clone Wan2.2 repo ────────────────────────────────────────────────────
import os

if not os.path.isdir("Wan2.2"):
    !git clone https://github.com/Wan-Video/Wan2.2.git
else:
    print("Wan2.2 directory already exists — skipping clone.")
    !cd Wan2.2 && git pull

Cloning into 'Wan2.2'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 266 (delta 77), reused 64 (delta 64), pack-reused 123 (from 2)
Receiving objects: 100% (266/266), 8.14 MiB | 40.45 MiB/s, done.
Resolving deltas: 100% (114/114), done.


In [ ]:
# ── 0.3  Install core dependencies ───────────────────────────────────────────
%cd Wan2.2
!pip install -q -r requirements.txt
!pip install -q .
!pip install -q decord
print("Core dependencies installed.")

/content/Wan2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 84.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.0 MB/s eta 0:00:00


In [ ]:
# ── 0.4  Install flash-attn (optional but recommended for speed) ──────────────
# This step can take 5-10 minutes. Skip if you hit issues — the model will
# fall back to standard attention automatically.
!pip install -q flash-attn --no-build-isolation
print("flash-attn installed.")

In [ ]:
# ── 0.5  Install S2V / CosyVoice dependencies (only needed for Section 4b) ───
# @markdown Run this cell only if you plan to use Speech-to-Video with TTS.
install_s2v = True  # @param {type:"boolean"}

if install_s2v:
    !pip install -q -r requirements_s2v.txt
    print("S2V/CosyVoice dependencies installed.")
else:
    print("Skipped S2V dependencies.")

In [ ]:
# ── 0.6  Install huggingface-hub CLI ─────────────────────────────────────────
!pip install -q "huggingface_hub[cli]"

# Optional: log in to HF to avoid rate limits on large model downloads
# from huggingface_hub import login
# login()   # uncomment and run if you have an HF token
print("huggingface_hub CLI ready.")

---
## Section 1 — Model Download

Download **only the model(s) you need**. Each model cell is independent.

| Model | Size (approx) | Use case |
|-------|--------------|----------|
| T2V-A14B | ~28 GB | Text → Video (480P / 720P) |
| I2V-A14B | ~28 GB | Image → Video (480P / 720P) |
| TI2V-5B  | ~10 GB | Text+Image → Video 720P@24fps, 4090-friendly |
| S2V-14B  | ~28 GB | Speech/Audio → Video |
| Animate-14B | ~28 GB | Character animation & replacement |

In [ ]:
# ── 1.1  Text-to-Video model — T2V-A14B ──────────────────────────────────────
import os
if not os.path.isdir("./Wan2.2-T2V-A14B"):
    !huggingface-cli download Wan-AI/Wan2.2-T2V-A14B --local-dir ./Wan2.2-T2V-A14B
else:
    print("T2V-A14B already downloaded.")

In [ ]:
# ── 1.2  Image-to-Video model — I2V-A14B ─────────────────────────────────────
import os
if not os.path.isdir("./Wan2.2-I2V-A14B"):
    !huggingface-cli download Wan-AI/Wan2.2-I2V-A14B --local-dir ./Wan2.2-I2V-A14B
else:
    print("I2V-A14B already downloaded.")

In [ ]:
# ── 1.3  Text+Image-to-Video model — TI2V-5B ─────────────────────────────────
# Smaller and faster — great for consumer GPUs (24 GB+)
import os
if not os.path.isdir("./Wan2.2-TI2V-5B"):
    !huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B
else:
    print("TI2V-5B already downloaded.")

In [ ]:
# ── 1.4  Speech-to-Video model — S2V-14B ─────────────────────────────────────
import os
if not os.path.isdir("./Wan2.2-S2V-14B"):
    !huggingface-cli download Wan-AI/Wan2.2-S2V-14B --local-dir ./Wan2.2-S2V-14B
else:
    print("S2V-14B already downloaded.")

In [ ]:
# ── 1.5  Character Animation model — Animate-14B ─────────────────────────────
import os
if not os.path.isdir("./Wan2.2-Animate-14B"):
    !huggingface-cli download Wan-AI/Wan2.2-Animate-14B --local-dir ./Wan2.2-Animate-14B
else:
    print("Animate-14B already downloaded.")

---
## Section 2 — Text-to-Video (T2V-A14B)

Generates video purely from a text prompt using the MoE 27B model (14B active parameters).
Supports **480P** and **720P** output.

In [ ]:
# ── 2.1  T2V Configuration ────────────────────────────────────────────────────
# @title Text-to-Video Settings

# @markdown ### Prompt
t2v_prompt = "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage."  # @param {type:"string"}

# @markdown ### Resolution
t2v_resolution = "1280*720"  # @param ["1280*720", "832*480"]

# @markdown ### Inference Steps & Seed
t2v_steps = 50  # @param {type:"integer"}
t2v_seed = 42   # @param {type:"integer"}

# @markdown ### Memory Optimization (enable if you hit OOM)
t2v_offload = True        # @param {type:"boolean"}
t2v_convert_dtype = True  # @param {type:"boolean"}
t2v_t5_cpu = False        # @param {type:"boolean"}

# @markdown ### Prompt Extension
# @markdown Leave `use_prompt_extend` off for fastest run. Enable to enrich prompt details.
t2v_use_prompt_extend = False     # @param {type:"boolean"}
t2v_extend_method = "local_qwen" # @param ["local_qwen", "dashscope"]
t2v_extend_model  = "Qwen/Qwen2.5-7B-Instruct"  # @param {type:"string"}
t2v_dashscope_key = ""           # @param {type:"string"}

# Build args string
t2v_args = ""
if t2v_offload:       t2v_args += "--offload_model True "
if t2v_convert_dtype: t2v_args += "--convert_model_dtype "
if t2v_t5_cpu:        t2v_args += "--t5_cpu "
if t2v_use_prompt_extend:
    t2v_args += f"--use_prompt_extend --prompt_extend_method '{t2v_extend_method}' "
    if t2v_extend_method == "local_qwen":
        t2v_args += f"--prompt_extend_model '{t2v_extend_model}' "

print("T2V configuration ready.")
print(f"  Resolution : {t2v_resolution}")
print(f"  Steps      : {t2v_steps}")
print(f"  Extra args : {t2v_args.strip()}")

In [ ]:
# ── 2.2  Run Text-to-Video Generation ────────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

_env = f"DASH_API_KEY={t2v_dashscope_key} " if t2v_dashscope_key else ""

!{_env}python generate.py \
    --task t2v-A14B \
    --size {t2v_resolution} \
    --ckpt_dir ./Wan2.2-T2V-A14B \
    --sample_steps {t2v_steps} \
    --base_seed {t2v_seed} \
    --prompt "{t2v_prompt}" \
    {t2v_args}

---
## Section 3 — Image-to-Video (I2V-A14B)

Animates a still image into a video. Supports **480P** and **720P**.
The output aspect ratio follows the input image's aspect ratio.

In [ ]:
# ── 3.1  Upload Input Image ───────────────────────────────────────────────────
from google.colab import files
import os

print("Upload the image you want to animate:")
uploaded = files.upload()

if uploaded:
    i2v_input_image = list(uploaded.keys())[0]
    print(f"Image set to: '{i2v_input_image}'")
else:
    # Fall back to the bundled example
    i2v_input_image = "examples/i2v_input.JPG"
    print(f"No upload detected — using example image: {i2v_input_image}")

In [ ]:
# ── 3.2  I2V Configuration ────────────────────────────────────────────────────
# @title Image-to-Video Settings

# @markdown ### Prompt  (leave empty to auto-generate from the image via prompt extension)
i2v_prompt = "Summer beach vacation style, a white cat wearing sunglasses sits on a surfboard, relaxed expression, crystal-clear waters in the background."  # @param {type:"string"}

# @markdown ### Resolution (area — aspect ratio follows the image)
i2v_resolution = "1280*720"  # @param ["1280*720", "832*480"]

# @markdown ### Inference Steps & Seed
i2v_steps = 50  # @param {type:"integer"}
i2v_seed  = 42  # @param {type:"integer"}

# @markdown ### Memory Optimization
i2v_offload       = True   # @param {type:"boolean"}
i2v_convert_dtype = True   # @param {type:"boolean"}
i2v_t5_cpu        = False  # @param {type:"boolean"}

# @markdown ### Prompt Extension  (auto-describes image when prompt is empty)
i2v_use_prompt_extend = False       # @param {type:"boolean"}
i2v_extend_method     = "local_qwen" # @param ["local_qwen", "dashscope"]
i2v_extend_model      = "Qwen/Qwen2.5-VL-7B-Instruct"  # @param {type:"string"}
i2v_dashscope_key     = ""          # @param {type:"string"}

i2v_args = ""
if i2v_offload:       i2v_args += "--offload_model True "
if i2v_convert_dtype: i2v_args += "--convert_model_dtype "
if i2v_t5_cpu:        i2v_args += "--t5_cpu "
if i2v_use_prompt_extend:
    i2v_args += f"--use_prompt_extend --prompt_extend_method '{i2v_extend_method}' "
    if i2v_extend_method == "local_qwen":
        i2v_args += f"--prompt_extend_model '{i2v_extend_model}' "

print("I2V configuration ready.")

In [ ]:
# ── 3.3  Run Image-to-Video Generation ───────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

if not os.path.exists(i2v_input_image):
    print(f"ERROR: Image '{i2v_input_image}' not found. Run cell 3.1 first.")
else:
    _env = f"DASH_API_KEY={i2v_dashscope_key} " if i2v_dashscope_key else ""
    !{_env}python generate.py \
        --task i2v-A14B \
        --size {i2v_resolution} \
        --ckpt_dir ./Wan2.2-I2V-A14B \
        --image "{i2v_input_image}" \
        --sample_steps {i2v_steps} \
        --base_seed {i2v_seed} \
        --prompt "{i2v_prompt}" \
        {i2v_args}

---
## Section 4 — Text+Image-to-Video (TI2V-5B)

Unified model supporting **both T2V and I2V** from a single 5B checkpoint.  
Uses the high-compression Wan2.2-VAE (16×16×4). Outputs **720P @ 24 FPS**.  
Runs on consumer GPUs with ≥ 24 GB VRAM (e.g. RTX 4090).

> Resolution for this model is `1280*704` (landscape) or `704*1280` (portrait).

In [ ]:
# ── 4.1  Upload Image (optional — omit for pure text-to-video) ───────────────
from google.colab import files
import os

# @markdown Set `use_image` to False for pure Text-to-Video with TI2V-5B.
ti2v_use_image = True  # @param {type:"boolean"}

ti2v_input_image = ""
if ti2v_use_image:
    print("Upload a reference image (or skip to use the bundled example):")
    uploaded = files.upload()
    if uploaded:
        ti2v_input_image = list(uploaded.keys())[0]
        print(f"Image: {ti2v_input_image}")
    else:
        ti2v_input_image = "examples/i2v_input.JPG"
        print(f"Using example: {ti2v_input_image}")
else:
    print("No image — running in Text-to-Video mode.")

In [ ]:
# ── 4.2  TI2V-5B Configuration ────────────────────────────────────────────────
# @title TI2V-5B Settings

ti2v_prompt     = "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage."  # @param {type:"string"}
ti2v_resolution = "1280*704"  # @param ["1280*704", "704*1280"]
ti2v_steps      = 50   # @param {type:"integer"}
ti2v_seed       = 42   # @param {type:"integer"}

# @markdown ### Memory Optimization (24 GB GPU: keep all True)
ti2v_offload       = True  # @param {type:"boolean"}
ti2v_convert_dtype = True  # @param {type:"boolean"}
ti2v_t5_cpu        = True  # @param {type:"boolean"}

ti2v_args = ""
if ti2v_offload:       ti2v_args += "--offload_model True "
if ti2v_convert_dtype: ti2v_args += "--convert_model_dtype "
if ti2v_t5_cpu:        ti2v_args += "--t5_cpu "
if ti2v_input_image:   ti2v_args += f'--image "{ti2v_input_image}" '

mode_label = "Image-to-Video" if ti2v_input_image else "Text-to-Video"
print(f"TI2V-5B mode: {mode_label}")

In [ ]:
# ── 4.3  Run TI2V-5B Generation ───────────────────────────────────────────────
import torch
torch.cuda.empty_cache()

!python generate.py \
    --task ti2v-5B \
    --size {ti2v_resolution} \
    --ckpt_dir ./Wan2.2-TI2V-5B \
    --sample_steps {ti2v_steps} \
    --base_seed {ti2v_seed} \
    --prompt "{ti2v_prompt}" \
    {ti2v_args}

---
## Section 5 — Speech-to-Video (S2V-14B)

Generates a talking-head / audio-driven video from:
- A **reference image** (portrait / character)
- An **audio file** (WAV/MP3 — speech, singing, etc.)
- An optional **text prompt**
- An optional **pose video** for full-body motion

Three sub-modes are provided:
- **5a**: Basic Speech-to-Video (image + audio)
- **5b**: Pose-driven Speech-to-Video (image + audio + pose video)
- **5c**: TTS-powered (CosyVoice synthesizes the audio for you)

In [ ]:
# ── 5.1  Upload Image & Audio ─────────────────────────────────────────────────
from google.colab import files
import os

print("── Upload reference image (portrait / character) ──")
up_img = files.upload()
if up_img:
    s2v_image = list(up_img.keys())[0]
else:
    s2v_image = "examples/i2v_input.JPG"
print(f"Image: {s2v_image}")

print("\n── Upload audio file (WAV or MP3) ──")
up_audio = files.upload()
if up_audio:
    s2v_audio = list(up_audio.keys())[0]
else:
    s2v_audio = "examples/talk.wav"
print(f"Audio: {s2v_audio}")

In [ ]:
# ── 5.2a  Basic Speech-to-Video Configuration ─────────────────────────────────
# @title S2V — Basic (image + audio)

s2v_prompt     = "A person speaking expressively."  # @param {type:"string"}
s2v_resolution = "1024*704"  # @param ["1024*704", "832*480", "1280*720"]
s2v_steps      = 40    # @param {type:"integer"}
s2v_seed       = 42    # @param {type:"integer"}

# @markdown `num_clip`: number of video clips (leave 0 to auto-match audio length)
s2v_num_clip   = 0     # @param {type:"integer"}

s2v_offload       = True  # @param {type:"boolean"}
s2v_convert_dtype = True  # @param {type:"boolean"}

s2v_args = ""
if s2v_offload:       s2v_args += "--offload_model True "
if s2v_convert_dtype: s2v_args += "--convert_model_dtype "
if s2v_num_clip > 0:  s2v_args += f"--num_clip {s2v_num_clip} "

print("S2V (basic) configuration ready.")

In [ ]:
# ── 5.2a  Run Basic Speech-to-Video ──────────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

for path, label in [(s2v_image, "image"), (s2v_audio, "audio")]:
    if not os.path.exists(path):
        print(f"ERROR: {label} file '{path}' not found. Upload in cell 5.1.")
        raise SystemExit()

!python generate.py \
    --task s2v-14B \
    --size {s2v_resolution} \
    --ckpt_dir ./Wan2.2-S2V-14B \
    --image "{s2v_image}" \
    --audio "{s2v_audio}" \
    --sample_steps {s2v_steps} \
    --base_seed {s2v_seed} \
    --prompt "{s2v_prompt}" \
    {s2v_args}

In [ ]:
# ── 5.3b  Pose-Driven Speech-to-Video ─────────────────────────────────────────
# @title S2V — Pose-Driven (image + audio + pose video)
# @markdown Upload a **pose video** (.mp4) that provides the body motion sequence.

from google.colab import files

print("Upload pose video (.mp4):")
up_pose = files.upload()
if up_pose:
    s2v_pose_video = list(up_pose.keys())[0]
else:
    s2v_pose_video = "examples/pose.mp4"
print(f"Pose video: {s2v_pose_video}")

# Reuse image/audio from 5.1 or override here:
# s2v_image = "examples/pose.png"
# s2v_audio = "examples/sing.MP3"

s2v_pose_prompt = "a person is singing"  # @param {type:"string"}

In [ ]:
# ── 5.3b  Run Pose-Driven Speech-to-Video ────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

!python generate.py \
    --task s2v-14B \
    --size {s2v_resolution} \
    --ckpt_dir ./Wan2.2-S2V-14B \
    --image "{s2v_image}" \
    --audio "{s2v_audio}" \
    --pose_video "{s2v_pose_video}" \
    --prompt "{s2v_pose_prompt}" \
    --offload_model True \
    --convert_model_dtype

In [ ]:
# ── 5.4c  TTS-powered Speech-to-Video (CosyVoice) ────────────────────────────
# @title S2V — TTS Mode  (requires S2V deps from cell 0.5)
# @markdown CosyVoice synthesizes audio from text, then Wan2.2-S2V animates the character.

# @markdown #### Reference audio for voice cloning (upload a short WAV of the target speaker)
from google.colab import files
print("Upload reference speaker audio for voice cloning:")
up_ref = files.upload()
if up_ref:
    tts_ref_audio = list(up_ref.keys())[0]
else:
    tts_ref_audio = "examples/zero_shot_prompt.wav"
print(f"Reference audio: {tts_ref_audio}")

# @markdown #### Text of the reference clip (transcription of the audio above)
tts_ref_text = "希望你以后能够做的比我还好呦。"  # @param {type:"string"}

# @markdown #### Text to synthesize and animate (the actual speech content)
tts_text = "收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。"  # @param {type:"string"}

tts_prompt = "A person giving an emotional speech."  # @param {type:"string"}

In [ ]:
# ── 5.4c  Run TTS Speech-to-Video ────────────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

!python generate.py \
    --task s2v-14B \
    --size {s2v_resolution} \
    --ckpt_dir ./Wan2.2-S2V-14B \
    --image "{s2v_image}" \
    --enable_tts \
    --tts_prompt_audio "{tts_ref_audio}" \
    --tts_prompt_text "{tts_ref_text}" \
    --tts_text "{tts_text}" \
    --prompt "{tts_prompt}" \
    --offload_model True \
    --convert_model_dtype

---
## Section 6 — Character Animation & Replacement (Animate-14B)

Wan-Animate takes a **reference character image** and a **source motion video**, and either:
- **Animation mode**: generates a new video of the character mimicking the motion.
- **Replacement mode**: replaces the character in the original video.

**Preprocessing is required** before inference. Two sub-sections below.

In [ ]:
# ── 6.1  Upload Assets for Animate ───────────────────────────────────────────
from google.colab import files
import os

print("── Upload reference character image (JPEG/PNG) ──")
up_char = files.upload()
animate_ref_image = list(up_char.keys())[0] if up_char else "examples/wan_animate/animate/image.jpeg"
print(f"Character image: {animate_ref_image}")

print("\n── Upload source motion video (MP4) ──")
up_vid = files.upload()
animate_src_video = list(up_vid.keys())[0] if up_vid else "examples/wan_animate/animate/video.mp4"
print(f"Motion video: {animate_src_video}")

In [ ]:
# ── 6.2  Choose Mode & Resolution ────────────────────────────────────────────
# @title Animate Settings

animate_mode = "animate"  # @param ["animate", "replace"]

# @markdown Resolution in pixels: width height (must be divisible by 16)
animate_res_w = 1280  # @param {type:"integer"}
animate_res_h = 720   # @param {type:"integer"}

# Preprocessing output directory
animate_proc_dir = f"./animate_proc_{animate_mode}"
os.makedirs(animate_proc_dir, exist_ok=True)

print(f"Mode: {animate_mode}")
print(f"Resolution: {animate_res_w}x{animate_res_h}")
print(f"Preprocessing output: {animate_proc_dir}")

In [ ]:
# ── 6.3  Preprocess Input Video ───────────────────────────────────────────────
# This step extracts poses, faces, and other signals needed by the model.
import os, torch
torch.cuda.empty_cache()

_use_flux = "--use_flux"
_retarget = "--retarget_flag" if animate_mode == "animate" else ""
_replace  = "--replace_flag"  if animate_mode == "replace"  else ""

print(f"Running preprocessing in '{animate_mode}' mode...")

if animate_mode == "animate":
    !python ./wan/modules/animate/preprocess/preprocess_data.py \
        --ckpt_path ./Wan2.2-Animate-14B/process_checkpoint \
        --video_path "{animate_src_video}" \
        --refer_path "{animate_ref_image}" \
        --save_path   {animate_proc_dir} \
        --resolution_area {animate_res_w} {animate_res_h} \
        --retarget_flag \
        --use_flux
else:
    !python ./wan/modules/animate/preprocess/preprocess_data.py \
        --ckpt_path ./Wan2.2-Animate-14B/process_checkpoint \
        --video_path "{animate_src_video}" \
        --refer_path "{animate_ref_image}" \
        --save_path   {animate_proc_dir} \
        --resolution_area {animate_res_w} {animate_res_h} \
        --iterations 3 \
        --k 7 \
        --w_len 1 \
        --h_len 1 \
        --replace_flag

In [ ]:
# ── 6.4a  Run Animate — Animation Mode ───────────────────────────────────────
import torch
torch.cuda.empty_cache()

if animate_mode == "animate":
    !python generate.py \
        --task animate-14B \
        --ckpt_dir ./Wan2.2-Animate-14B \
        --src_root_path {animate_proc_dir} \
        --refert_num 1
else:
    print("Mode is 'replace' — run cell 6.4b instead.")

In [ ]:
# ── 6.4b  Run Animate — Replacement Mode ─────────────────────────────────────
import torch
torch.cuda.empty_cache()

if animate_mode == "replace":
    !python generate.py \
        --task animate-14B \
        --ckpt_dir ./Wan2.2-Animate-14B \
        --src_root_path {animate_proc_dir} \
        --refert_num 1 \
        --replace_flag \
        --use_relighting_lora
else:
    print("Mode is 'animate' — run cell 6.4a instead.")

---
## Section 7 — Display & Download Output

Find, display, and download the most recently generated video.

In [ ]:
# ── 7.1  List all generated videos ────────────────────────────────────────────
import glob, os

mp4_files = sorted(glob.glob("**/*.mp4", recursive=True), key=os.path.getctime, reverse=True)
if mp4_files:
    print(f"Found {len(mp4_files)} video(s):")
    for i, f in enumerate(mp4_files):
        size_mb = os.path.getsize(f) / 1e6
        print(f"  [{i}] {f}  ({size_mb:.1f} MB)")
else:
    print("No .mp4 files found. Generate a video first.")

In [ ]:
# ── 7.2  Preview the latest video ─────────────────────────────────────────────
import glob, os
from IPython.display import Video, display

mp4_files = sorted(glob.glob("**/*.mp4", recursive=True), key=os.path.getctime, reverse=True)

preview_index = 0  # @param {type:"integer"}

if mp4_files and preview_index < len(mp4_files):
    chosen = mp4_files[preview_index]
    print(f"Previewing: {chosen}")
    display(Video(chosen, embed=True, width=854, height=480))
else:
    print("No video to display.")

In [ ]:
# ── 7.3  Download a video to your computer ────────────────────────────────────
import glob, os
from google.colab import files as colab_files

mp4_files = sorted(glob.glob("**/*.mp4", recursive=True), key=os.path.getctime, reverse=True)

download_index = 0  # @param {type:"integer"}

if mp4_files and download_index < len(mp4_files):
    chosen = mp4_files[download_index]
    print(f"Downloading: {chosen}")
    colab_files.download(chosen)
else:
    print("No video to download.")